In [ ]:
# https://github.com/danielgatis/rembg
# https://arxiv.org/pdf/2005.09007.pdf

!pip install rembg

In [ ]:
from rembg import remove
import requests
from PIL import Image
from io import BytesIO
import os

In [ ]:
os.makedirs('original', exist_ok=True)
os.makedirs('masked', exist_ok=True)

In [ ]:
img_url = 'https://nationaltoday.com/wp-content/uploads/2020/12/National-Horse-Day-1-640x514.jpg'
img_name = img_url.split('/')[-1]
img_name

In [ ]:
img = Image.open(BytesIO(requests.get(img_url).content))
img.save('original/'+img_name, format='jpeg')

In [ ]:
output_path = 'masked/'+img_name
output_path

In [ ]:
with open(output_path, 'wb') as f:
  input = open('original/'+img_name, 'rb').read()
  subject = remove(input, alpha_matting=True, alpha_matting_foreground_threshold=50)
  f.write(subject)

In [ ]:
background_img = 'https://iso.500px.com/wp-content/uploads/2014/07/big-one.jpg'
background_img = Image.open(BytesIO(requests.get(background_img).content))

background_img = background_img.resize((img.width, img.height))

foreground_img = Image.open(output_path)
background_img.paste(foreground_img, (0,0), foreground_img)
background_img.save('masked/background.jpg', format='jpeg')

In [ ]:
!pip install streamlit

In [ ]:
!pip install pyngrok

In [ ]:
from pyngrok import ngrok

# Start Streamlit app
!streamlit run app.py &

# Create a public URL using ngrok
public_url = ngrok.connect(port='8501')

# Print the public URL
print('Streamlit app is live at:', public_url)

In [ ]:
subject_url = "https://images.squarespace-cdn.com/content/v1/5b97966c710699158dadbec0/1607591534466-7L1NE41JFQLNO9TDJQ6P/Make+My+Day+1.jpg"
background_url = "https://images.all-free-download.com/images/graphicwebp/park_autumn_garden_219181.webp"

subject_name = subject_url.split('/')[-1]
background_name = background_url.split('/')[-1]

     # Save images to the "original" folder
original_folder = 'original'
subject_file = os.path.join(original_folder, subject_name)
background_file = os.path.join(original_folder, background_name)


In [ ]:
subject_file

In [ ]:
background_file

In [ ]:
import streamlit as st
from PIL import Image
import os
import requests
from io import BytesIO
from rembg import remove

# set full screen
st.set_page_config(layout="wide")

st.title("Image Background Removal and Replacement using ML and AI")
st.write("This is a simple image background removal and replacement web app using ML and AI")

os.makedirs('original', exist_ok=True)
os.makedirs('masked', exist_ok=True)

use_local_image = st.checkbox("Use Local Image", value=False)
cols = st.columns(2)
subject_file = None
background_file = None
subject_name = None
background_name = None  # Define subject_name and background_name

if use_local_image:
    # Upload Image
    st.header("Upload Image")
    cols = st.columns(2)

    subject_file = cols[0].file_uploader("Choose Subject Image...", type=["jpg", 'png', 'jpeg'], key='subject')
    background_file = cols[1].file_uploader("Choose Background Image...", type=["jpg", 'png', 'jpeg'], key='background')

    if subject_file is not None:
        subject_name = subject_file.name
        subject_file_data = subject_file.read()
        subject_file_path = os.path.join('original', subject_name)
        with open(subject_file_path, 'wb') as f:
            f.write(subject_file_data)
        subject_img = Image.open(BytesIO(subject_file_data))


    if background_file is not None:
        background_name = background_file.name
        background_file_data = background_file.read()
        background_file_path = os.path.join('original', background_name)
        with open(background_file_path, 'wb') as f:
            f.write(background_file_data)
        background_img = Image.open(BytesIO(background_file_data))

# Display selected images
    if subject_name and background_name:
        st.title("Selected Images")
        cols = st.columns(2)

        cols[0].image(subject_img, caption='Subject Image', use_column_width=True)
        cols[1].image(background_img, caption='Background Image', use_column_width=True)

        # Image processing
        st.title("Removing Background from Subject Image and Replacing it with Background Image")
        threshold = st.slider("Background Threshold", 0, 255, value=50, step=5)

        cols = st.columns(2)

        output_file = "masked/" + subject_name
        f = open(output_file, 'wb')
        subject_img_data = open(subject_file, 'rb').read()
        subject = remove(subject_img_data, alpha_matting=True, alpha_matting_foreground_threshold=threshold)
        f.write(subject)
        f.close()

        cols[0].image(output_file, caption="Subject Image without Background. Use Slider to Control Background Removal", use_column_width=True)

        background_img = Image.open(background_file)
        subject_img = Image.open(output_file)

        background_img = background_img.resize(subject_img.size)

        background_img.paste(subject_img, (0,0), subject_img)
        background_img.save('masked/background.jpg', format='jpeg')

        cols[1].image('masked/background.jpg', caption='Merged Image', use_column_width=True)

    else:
        st.warning("Please select both the subject and background images.")



else:
    subject_url = cols[0].text_input("Enter Subject Image URL", "https://images.squarespace-cdn.com/content/v1/5b97966c710699158dadbec0/1607591534466-7L1NE41JFQLNO9TDJQ6P/Make+My+Day+1.jpg")
    background_url = cols[1].text_input("Enter Background URL", "https://images.all-free-download.com/images/graphicwebp/park_autumn_garden_219181.webp")

    subject_name = subject_url.split('/')[-1]
    background_name = background_url.split('/')[-1]

    subject_file = os.path.join('original', subject_name)
    background_file = os.path.join('original', background_name)

    try:

        subject_img = Image.open(BytesIO(requests.get(subject_url).content))
        subject_img.save(subject_file, format='jpeg')

        background_img = Image.open(BytesIO(requests.get(background_url).content))
        background_img.save(background_file, format='jpeg')
    except Exception as e:
        pass

    cols[0].image(subject_img, caption='Subject Image', use_column_width=True)

    background_img = Image.open(background_file)
    cols[1].image(background_img, caption='Background Image', use_column_width=True)

    st.title("Removing Background from Subject Image and Replacing it with Background Image")
    threshold = st.slider("Background Threshold", 0, 255, value=50, step=5)

    cols = st.columns(2)

    output_file = "masked/" + subject_name
    f = open(output_file, 'wb')
    subject_img = open(subject_file, 'rb').read()
    subject = remove(subject_img, alpha_matting=True, alpha_matting_foreground_threshold=threshold)
    f.write(subject)
    f.close()

    cols[0].image(output_file, caption="Subject Image without Background. Use Slider to Control Background Removal", use_column_width=True)

    background_img = Image.open(background_file)
    subject_img = Image.open(output_file)

    background_img = background_img.resize(subject_img.size)

    background_img.paste(subject_img, (0,0), subject_img)
    background_img.save('masked/background.jpg', format='jpeg')

    cols[1].image('masked/background.jpg', caption='Merged Image', use_column_width=True)
